# Multiple Linear Regression on Hydrologic Data to predict Baseflow

## Setup

In [ ]:
# Standard Library
from enum import StrEnum

# 3rd party
from pandas import read_csv, DataFrame, to_datetime, Timestamp
from seaborn import displot, scatterplot
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

DATA_PATH: str = "./data/RRCA_baseflow.csv"

class FlowCol(StrEnum):
    """
    Thanks Toby for this idea
    Lets you have auto-complete for the headers,
    less magic strings.
    """
    # Original Column Names
    DATE='Date',
    ID='Segment_id',
    X='x',
    Y='y',
    EVAPOTRANSPIRATION='Evapotranspiration',
    PRECIP='Precipitation',
    IRRIGATION_PUMPING='Irrigation_pumping',
    BASEFLOW='Observed' # The observed baseflow, I thought it made more sense to
                        # to name it BASEFLOW
    
    # Column Names Added In
    READABLE_DATE='readable_date',
    DAY='day',
    MONTH='month',
    YEAR='year'

baseflow_data: DataFrame = read_csv(DATA_PATH)

## Data Preparation

### Fix Dates

Adjust dates to be days since Jan 1st 1900, rather than previous value of days since Jan 1st 0000.

In [ ]:
TEST_INDEX:int = 15586
date_before:int = baseflow_data[FlowCol.DATE][TEST_INDEX]

DATE_ADJUSTMENT: int = 693963 # value from proj instructions

baseflow_data[FlowCol.DATE] = (
    [date - DATE_ADJUSTMENT for date in baseflow_data[FlowCol.DATE]]
)

date_after:int = baseflow_data[FlowCol.DATE][TEST_INDEX]
assert(date_before - date_after == DATE_ADJUSTMENT)

Using professor's example to get more readable dates from the previous Date values

In [ ]:
ORIGIN_DATE:Timestamp = Timestamp("1900-01-01")
baseflow_data[str(FlowCol.READABLE_DATE)] = to_datetime(
    baseflow_data[FlowCol.DATE].astype('int'), 
    unit='D', 
    origin=ORIGIN_DATE
)

baseflow_data[str(FlowCol.YEAR)] = (
    baseflow_data[FlowCol.READABLE_DATE].apply(lambda x:to_datetime(x).year)
)
baseflow_data[str(FlowCol.MONTH)] = (
    baseflow_data[FlowCol.READABLE_DATE].apply(lambda x:to_datetime(x).month)
)
baseflow_data[str(FlowCol.DAY)] = (
    baseflow_data[FlowCol.READABLE_DATE].apply(lambda x:to_datetime(x).day)
)

display(baseflow_data.head())

## Ideas to try next for making a better model

* only include gaging stations that have at least 10 years of monthly baseflow observations
* remove streamflow losses (non positive baseflow conditions)
* remove time periods that corresponde to zero baseflow
* use data from 1941-1989 for training and to fit the error distribution, and use data from 1990-2000 for validation.

Remove Streamflow Losses (non positive baseflow conditions) and Remove time periods that correspond to zero baseflow

In [ ]:
baseflow_data = baseflow_data[baseflow_data[FlowCol.BASEFLOW] > 0.0]

#Create a subset of the rows that are being dropped (baseflow <= 0)
weeded_out = baseflow_data[baseflow_data[FlowCol.BASEFLOW] <= 0.0]

# Total number of entries removed
print(f"Total entries removed: {len(weeded_out)}")
print(f"Percentage of dataset removed: {(len(weeded_out) / len(baseflow_data)) * 100:.2f}%")

# Break down into exact zeros vs. actual negative numbers
num_zeros = (weeded_out[FlowCol.BASEFLOW] == 0.0).sum()
num_negatives = (weeded_out[FlowCol.BASEFLOW] < 0.0).sum()
print(f"Exact zeros: {num_zeros}")
print(f"Negative entries: {num_negatives}")

Only include gaging stations that have at least 10 years of monthly baseflow observations

In [ ]:
MIN_YRS = 10

# display(baseflow_data[baseflow_data[FlowCol.ID] == 144])
# display(baseflow_data[baseflow_data[FlowCol.ID] == 144][FlowCol.YEAR].unique())
# display(len(baseflow_data[baseflow_data[FlowCol.ID] == 144][FlowCol.YEAR].unique()))

print(f"Data size before: {baseflow_data.size}")

ids = baseflow_data[FlowCol.ID].unique()
for id in ids:
    if (len(baseflow_data[baseflow_data[FlowCol.ID] == id][FlowCol.YEAR].unique()) < MIN_YRS):
        baseflow_data = baseflow_data[baseflow_data[FlowCol.ID] != id]

print(f"Data size after: {baseflow_data.size}")

Split data into Training data (1939-1989) and Test data (1990-2000)

In [ ]:

# YEAR_THRESHOLD = 1989

# display(baseflow_data[FlowCol.YEAR].min())
# display(baseflow_data[FlowCol.YEAR].max())

# # TODO I just included all the years, did you mean specifically to exclude years less than 1941?
# # Just saw in the dataset there is at least one measurement for 1939
# baseflow_data_train = baseflow_data[baseflow_data[FlowCol.YEAR] <= YEAR_THRESHOLD]
# baseflow_data_test = baseflow_data[baseflow_data[FlowCol.YEAR] > YEAR_THRESHOLD]

# display(baseflow_data_train.size)
# display(baseflow_data_test.size)

# we decided not to do this because we want to use all the data, and we don't want to exclude any years. Linear regression doesn't need to be trained, unlike what they were doing in the paper.


## Analysis

### Evapotranspiration with Baseflow

In [ ]:
scatterplot(data=baseflow_data, x=FlowCol.EVAPOTRANSPIRATION, y=FlowCol.BASEFLOW)

### Precipitation with Baseflow

In [ ]:
scatterplot(data=baseflow_data, x=FlowCol.PRECIP, y=FlowCol.BASEFLOW)

### Irrigation Pumping with Baseflow

In [ ]:
scatterplot(data=baseflow_data, x=FlowCol.IRRIGATION_PUMPING, y=FlowCol.BASEFLOW)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.scatterplot(
    data=baseflow_data,
    x=baseflow_data[FlowCol.IRRIGATION_PUMPING].apply(np.log1p), # Use np.log1p if there are zeros
    y=FlowCol.BASEFLOW
)

plt.xlabel("Log Irrigation Pumping")
plt.show()

# conclusion from this is that this is not logarithmic, it's actually linear but it has a ton of 0s

### Baseflow Over Time

In [ ]:
scatterplot(data=baseflow_data, x=FlowCol.READABLE_DATE, y=FlowCol.BASEFLOW)

### Irrigation pumping over time

In [ ]:
scatterplot(data=baseflow_data, x=FlowCol.READABLE_DATE, y=FlowCol.IRRIGATION_PUMPING)

In [ ]:
# Baseflow over the months
scatterplot(data=baseflow_data, x=FlowCol.MONTH, y=FlowCol.BASEFLOW)

In [ ]:
len(baseflow_data[FlowCol.ID].unique())


In [ ]:
# I think this will be what we do, at least something like this,
# eventually once we have decided that data is cleaned up enough. Just wanted
# to start it here so there is at least something

# We probably will need to do some sort of analysis with the 
# dates as well
explanatory_vars = [
    str(FlowCol.EVAPOTRANSPIRATION), 
    str(FlowCol.PRECIP), 
    str(FlowCol.IRRIGATION_PUMPING)
]

explanatories = baseflow_data[explanatory_vars]
target = baseflow_data[FlowCol.BASEFLOW]

lm = LinearRegression()
lm.fit(explanatories, target)


for val in zip(explanatory_vars, lm.coef_):
    print(val)

display(lm.intercept_)
display(lm.coef_)

# Print summary statistics to compare magnitudes and units
cols_to_check = [
    str(FlowCol.EVAPOTRANSPIRATION), 
    str(FlowCol.PRECIP), 
    str(FlowCol.IRRIGATION_PUMPING),
    str(FlowCol.BASEFLOW)
]

print(baseflow_data[cols_to_check].describe())


In [ ]:
# same model as above but using statsmodels so we can get a summary of the model
import statsmodels.formula.api as smf

formula_str = f"{FlowCol.BASEFLOW} ~ {FlowCol.EVAPOTRANSPIRATION} + {FlowCol.PRECIP} + {FlowCol.IRRIGATION_PUMPING}"

lm = smf.ols(formula=formula_str, data=baseflow_data).fit()

display(lm.params)

# print a summary of the fitted model
lm.summary()

### The winning model

In [ ]:
# trying to do a log transformation of the baseflow, since it is skewed
# this is the best model, it is the one we will use for the final model
formula_str = (
    f"np.log({FlowCol.BASEFLOW}) ~  "
    f"{FlowCol.EVAPOTRANSPIRATION} + "
    f"{FlowCol.PRECIP} + "
    f"{FlowCol.IRRIGATION_PUMPING}"
)

lm_improved = smf.ols(formula=formula_str, data=baseflow_data).fit()

print(lm_improved.summary())

In [ ]:
# trying to add year as features to see if that improves the model
# It does not improve the model, so we will not use it in the final model
formula_str = (
    f"np.log({FlowCol.BASEFLOW}) ~ "
    f"{FlowCol.EVAPOTRANSPIRATION} + "
    f"{FlowCol.PRECIP} + "
    f"{FlowCol.IRRIGATION_PUMPING} + "
    f"{FlowCol.YEAR} "
)

lm_improved = smf.ols(formula=formula_str, data=baseflow_data).fit()

print(lm_improved.summary())